In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_timestamp, current_timestamp, last, lag, abs
from pyspark.sql.window import Window
import random
from datetime import datetime, timedelta

spark = SparkSession.builder \
    .appName("Clean Sensor Data") \
    .getOrCreate()

spark

In [18]:
def test_data_minimal():
    return [{
            "sensor_id": 1,
            "timestamp": "2025-11-20T21:00:00",
            "temperature": 25.0,
            "humidity": 40.0,
            "co2": 501
            },
            {
            "sensor_id": 2,
            "timestamp": "2026-11-20T21:00:00",
            "temperature": 25.0,
            "humidity": 40.0,
            "co2": 501
            },
            {
            "sensor_id": 3,
            "timestamp": "2025-11-20T21:00:00",
            "temperature": 30.0,
            "humidity": 40.0,
            "co2": 501
            },
            {
            "sensor_id": 1,
            "timestamp": "2025-12-20T21:00:00",
            "temperature": 35.0,
            "humidity":None,
            "co2": 520
            },
            {
           "sensor_id": 2,
            "timestamp": "2025-12-20T21:00:00",
            "temperature": 18.0,
            "humidity": 20.0,
            "co2": 1900
            },
            {
           "sensor_id": 3,
            "timestamp": "2025-12-20T21:00:00",
            "temperature": 18.0,
            "humidity": 20.0,
            "co2": 1900
            },
            {
           "sensor_d": 3,
            "timestamp": "2025-11-20T21:00:00",
            "temperature": 18.0,
            "humidity": 20.0,
            "co2": 1900
            }
           ]

def test_data_randomized_small(NUM_SENSORS,NUM_STEPS):
    sensor_state = {
    sensor_id: {
        "temperature": random.uniform(18.0, 23.0),
        "humidity": random.uniform(40.0, 60.0),
        "co2": random.uniform(400, 4600)
    }
    for sensor_id in range(1, NUM_SENSORS + 1)
}
    def update_value(value,min_val,max_val,max_change):
        change=random.uniform(-max_change,max_change)
        new_value=value+change
        return max(min(new_value,max_val),min_val)

    def generate_sensor_data():
        data=[]
        start_time=datetime.now()


        for step in range(NUM_STEPS):
            timestamp=start_time+timedelta(seconds=step)

            for sensor_id,state in sensor_state.items():
                state["temperature"]=update_value(state["temperature"], -15,60,0.3)
                state["humidity"]=update_value(state["humidity"], -5,110,1)
                state["co2"]=update_value(state["co2"], 300,6000,10)


                record={
                    "sensor_id":sensor_id,
                    "timestamp":timestamp.isoformat(),
                    "temperature":round(state["temperature"],2),
                    "humidity":round(state["humidity"],2),
                    "co2":int(state["co2"])
                }


                if random.random() <0.10:
                    record["humidity"]+=12
                    
                if random.random() < 0.05:
                    record["humidity"] = None

                if random.random() <0.10:
                    record["temperature"]+=6
                    
                if random.random() <0.10:
                    record["co2"]+=500

                
                data.append(record)

                if random.random() < 0.02:
                    data.append(record.copy())
        return data


    return generate_sensor_data()
    

In [24]:

def clean_data(data):
    
    df=spark.createDataFrame(data)
   
    schemad_df=(
        df.select(
            col("sensor_id").cast("int"),
            to_timestamp(col("timestamp"), "yyyy-MM-dd'T'HH:mm:ss.SSSSSS").alias("timestamp"),
            col("temperature").cast("double"),
            col("humidity").cast("double"),
            col("co2").cast("int"),
            )
    )
    
    df_deduped=schemad_df.dropDuplicates(subset=["sensor_id", "timestamp"])

    window_spec_fill=(
        Window
        .partitionBy("sensor_id")
        .orderBy("timestamp")
        .rowsBetween(Window.unboundedPreceding, Window.currentRow)
    )

    window_spec_anomaly=(
        Window
        .partitionBy("sensor_id")
        .orderBy("timestamp")
    )
    FILL_COLUMNS=["temperature","humidity","co2"]
    for c in FILL_COLUMNS:
        df_deduped=df_deduped.withColumn(
            c,
            last(col(c),ignorenulls=True).over(window_spec_fill)
        )

    
    df_no_null_ranges=df_deduped.filter(
        col("sensor_id").isNotNull() &
        col("timestamp").isNotNull() &
        (col("timestamp") <= current_timestamp()) &
        col("temperature").between(-10,50) &
        col("humidity").between(0,100) &
        col("co2").between(350,5000)
    )

    df_deltas=df_no_null_ranges.withColumns({
        "temp_delta": abs(col("temperature") -lag("temperature").over(window_spec_anomaly)),
        "humid_delta": abs(col("humidity") -lag("humidity").over(window_spec_anomaly)),
        "co2_delta": abs(col("co2") -lag("co2").over(window_spec_anomaly))
    })


    df_anomaly=df_deltas.withColumns({
        "temp_anomaly": col("temp_delta")>3,
        "humid_anomaly": col("humid_delta")>4,
        "co2_anomaly": col("co2_delta")>100
    })

    df_anomaly=df_anomaly.withColumn(
        "anomaly",
        col("temp_anomaly") |
        col("humid_anomaly") |
        col("co2_anomaly")
    )

    return df_anomaly
    
    

In [25]:
test_data=test_data_randomized_small(10,100)
df=clean_data(test_data)

In [26]:
df.filter(
    col("temp_anomaly")==True
).show(100,truncate=False)

+---------+--------------------------+-----------+--------+----+-----------------+-------------------+---------+------------+-------------+-----------+-------+
|sensor_id|timestamp                 |temperature|humidity|co2 |temp_delta       |humid_delta        |co2_delta|temp_anomaly|humid_anomaly|co2_anomaly|anomaly|
+---------+--------------------------+-----------+--------+----+-----------------+-------------------+---------+------------+-------------+-----------+-------+
|1        |2026-01-21 15:38:06.245115|26.4       |59.33   |657 |6.02             |0.9399999999999977 |4        |true        |false        |false      |true   |
|1        |2026-01-21 15:38:07.245115|20.48      |58.93   |657 |5.919999999999998|0.3999999999999986 |0        |true        |false        |false      |true   |
|5        |2026-01-21 15:38:06.245115|26.03      |43.14   |2776|6.130000000000003|0.0                |5        |true        |false        |false      |true   |
|5        |2026-01-21 15:38:07.245115|20